In [73]:
import pandas as pd
import pyreadstat
# set pandas display options to show all columns
pd.set_option('display.max_columns', None)

# import basics
import pandas as pd
import numpy as np

# import tools
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin

# import models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from lightgbm import LGBMRegressor

# import viz
import altair as alt
alt.renderers.enable('mimetype') # for altair plots to be properly rendered on GH


RendererRegistry.enable('mimetype')

In [74]:
df_school, meta_school = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_SCHOOL16.sav')
df_student, meta_student = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STUDENT16.sav')
df_teacher, meta_teacher = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_TEACHER16.sav')
df_link, meta_link = pyreadstat.read_sav('../data/Data_PIRLS16(sav)/P4_STD_TCH_LINK16.sav')


In [75]:
# cols to generate y
y_gen_cols = ['ASRREA01', 'ASRREA02', 'ASRREA03', 'ASRREA04', 'ASRREA05']
# cols to drop because they might lead to leakage
cols_drop = y_gen_cols + ['ASRLIT01', 'ASRLIT02', 'ASRLIT03', 'ASRLIT04', 'ASRLIT05', 'ASRINF01', 'ASRINF02', 'ASRINF03', 'ASRINF04', 'ASRINF05', 'ASRIIE01', 'ASRIIE02', 'ASRIIE03', 'ASRIIE04', 'ASRIIE05', 'ASRRSI01', 'ASRRSI02', 'ASRRSI03', 'ASRRSI04', 'ASRRSI05', ]

In [76]:
def create_y(df, y_gen_cols):
    """Create y column by averaging the columns in y_gen_cols"""
    df['y'] = df[y_gen_cols].mean(axis=1)
    return df

def remove_cols(df, cols_drop):
    """Remove columns from the dataframe if they exist"""
    # drop column if it exists
    to_drop = [col for col in cols_drop if col in df.columns]
    # drop columns
    df = df.drop(columns=to_drop)
    return df

def merge_dfs(df_student, df_new, on):
    """ Merge df_new into df_student on the given column"""
    # record original number of rows
    original_rows = df_student.shape[0]
    # drop overlapping columns in df_new except for the merge column
    cols_to_drop = [col for col in df_new.columns if col in df_student.columns and col != on]
    df_new = df_new.drop(columns=cols_to_drop)
    # merge the dataframes
    df_student = df_student.merge(df_new, on=on, how='left', suffixes=('', '_new'))
    # drop the new columns that are now duplicates
    cols_to_drop = [col for col in df_student.columns if col.endswith('_new')]
    df_student = df_student.drop(columns=cols_to_drop)
    # assert that the number of rows is the same
    assert df_student.shape[0] == original_rows, f"Number of rows changed from {original_rows} to {df_student.shape[0]}"
    return df_student


In [77]:
df_student.shape

(4425, 152)

In [78]:
def create_X_and_y(df_student, df_teacher, df_school, df_link):
    """ Create a dataframe with the relevant columns from the student, teacher, school, and link dataframes"""
    global y_gen_cols, cols_drop
    df_student = create_y(df_student, y_gen_cols)
    df_student = remove_cols(df_student, cols_drop)
    df_school = remove_cols(df_school, cols_drop)
    df_teacher = remove_cols(df_teacher, cols_drop)
    df_link = remove_cols(df_link, cols_drop)
    
    df = merge_dfs(df_student, df_school, on='IDSCHOOL')
    df = merge_dfs(df, df_link, on='IDSTUD')
    df = merge_dfs(df, df_teacher, on='IDTEALIN')
    
    y = df['y']
    X = df.drop(columns=['y'])
    return X, y

In [79]:
X, y = create_X_and_y(df_student, df_teacher, df_school, df_link)
X.shape, y.shape

((4425, 390), (4425,))

In [80]:
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y,
                                                    test_size=0.3,
                                                    random_state=9527)


In [ ]:
drop_cols = []
# create the list of cols to drop based on missing rate
missing_rates = (X_train.isnull().sum()/X_train.shape[0]).sort_values(ascending=False)[:20]
miss_rate_bar = 0.8
missing_cols = missing_rates[missing_rates > miss_rate_bar].index.tolist()
drop_cols += missing_cols

# id columns
id_cols = [col for col in X_train.columns if 'ID' in col]
drop_cols += id_cols

# date column
drop_cols += ['ITDATE']

# get list of columns that have only one unique value and with no missing values
def get_constant_cols(X):
    """ Get the constant columns from the dataframe"""
    const_cols = X.nunique()[X.nunique() == 1].index.tolist()
    # get the columns that have no missing values
    const_cols = [col for col in const_cols if X[col].isnull().sum() == 0]
    return const_cols

const_cols = get_constant_cols(X_train)
drop_cols += const_cols

# X_train.drop(columns=missing_cols, inplace=True)

# a custom sklearn pipeline function step to remove columns that are missing too much data
class ColumnDropper(BaseEstimator, TransformerMixin):
    def __init__(self, drop_cols):
        self.drop_cols = drop_cols
        self.is_fitted = False
        pass
    
    def fit(self, X, y=None):
        self.is_fitted = True
        return self

    def transform(self, X):
        assert self.is_fitted, "The ColumnDropper has not been fitted yet. Please call fit() before transform()."
        X_drop = X.drop(columns=self.drop_cols)
        self.remaining_cols = X_drop.columns
        return X_drop

In [97]:
models = {"Dummy": DummyRegressor(), 
        #  "LinearReg": LinearRegression(),
         "DT": DecisionTreeRegressor(),
        #  "RF": RandomForestRegressor(),
        #  "SVM": SVR(),
         "LGBM": LGBMRegressor()}

preproc = ColumnDropper(drop_cols)
# ppl = Pipeline(steps=[('preprocessor', preproc),
#                       ('regressor', model)])
results = {}

In [98]:
# run all the models with 10-fold CV, RMSE scoring
for model_name, model in models.items():
    ppl = Pipeline(steps=[('preprocessor', preproc),
                      ('regressor', model)]) # create pipeline object
    ppl.fit(X_train, y_train)
    score = cross_val_score(ppl, X_train, y_train, scoring='neg_mean_absolute_error', cv=10).mean() * -1 # 5-fold cv MAE score
    
    print("{}: {:.3f}".format(model_name, score))
    
    results[model_name] = score # record model's performance


Dummy: 61.284
DT: 13.892
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003083 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4867
[LightGBM] [Info] Number of data points in the train set: 3097, number of used features: 367
[LightGBM] [Info] Start training from score 548.731639
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002746 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4849
[LightGBM] [Info] Number of data points in the train set: 2787, number of used features: 366
[LightGBM] [Info] Start training from score 547.920743
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003370 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4847
[LightGBM] [Info] Number of data points in the train set: 2787, number of used features: 36

In [99]:
score

9.140383060753262

In [104]:
# visualize LGBM's predictions agains y_true on test set
df_viz = pd.DataFrame({"id":range(len(X_test)), 
                       "pred":ppl.predict(X_test),
                      "true": y_test})

df_viz['pred-true'] = df_viz['pred'] - df_viz['true']

# melt for viz
df_viz_melt = df_viz.melt(id_vars='id', 
           var_name='type',
            value_vars=['pred', 'true'],
            value_name='y'
           )


In [106]:
# line plot of y_pred and y_true
plot_line = alt.Chart(df_viz_melt).mark_line(opacity=0.4).encode(
    x='id',
    y='y',
    color='type'
)

# bar plot of diff between pred and true
plot_bar = alt.Chart(df_viz).mark_bar(width=2).encode(
    x=alt.X('id', scale=alt.Scale(domain=(10, 400))),
#     x='id',
    y='pred-true'
)

# aggregated plot
(plot_line + plot_bar).properties(
    width=1300,
    height=300,
    title="y_pred vs y_true, with diff in bars"
).interactive()


<VegaLite 5 object>

If you see this message, it means the renderer has not been properly enabled
for the frontend that you are using. For more information, see
https://altair-viz.github.io/user_guide/display_frontends.html#troubleshooting


In [ ]:
ftr_imptns = ppl['regressor'].feature_importances_


In [108]:
ppl['preprocessor']

ColumnDropper(drop_cols=['ASBG05H', 'ASBGHRL', 'ASDGHRL', 'ASBGDDH', 'ASDGDDH',
                         'ASXBG03Ba', 'IDCNTRY', 'IDBOOK', 'IDSCHOOL',
                         'IDCLASS', 'IDSTUD', 'IDGRADE', 'IDPOP', 'IDGRADER',
                         'IDTEACH', 'IDLINK', 'IDTEALIN', 'IDSUBJ', 'ITDATE',
                         'IDCNTRY', 'IDGRADE', 'ITADMINI', 'ITLANG', 'IDPOP',
                         'IDGRADER', 'WGTADJ2', 'IDSUBJ', 'NTEACH'])

In [ ]:
# visualize LGBM important features

ftr_imptns_df = pd.DataFrame({"feature": mdl_col_names,
             "importance": ftr_imptns})
ftr_imptns_df = ftr_imptns_df.sort_values('importance', ascending = False).reset_index(drop=True)

n = 30 # how many top features to display in the plot
bars = alt.Chart(ftr_imptns_df.iloc[:n, :]).mark_bar().encode(
    x='importance:Q',
    y=alt.Y('feature:O', sort='-x')
)
bars.properties(
    title="LGBM - feature importance (top30)"
)
